# iprPy energy_check calculation

In [1]:
# Standard library imports
import datetime

# http://www.numpy.org/
import numpy as np

# https://ipython.org/
from IPython.display import display, Code, Markdown, Pretty

# https://github.com/usnistgov/atomman
import atomman as am
import atomman.unitconvert as uc

# https://github.com/usnistgov/iprPy
import iprPy

print('Notebook last executed on', datetime.date.today(), 'using iprPy version', iprPy.__version__)

Notebook last executed on 2026-06-24 using iprPy version 0.12.a


## 1. Load calculation and view description

### 1.1. Load the calculation

In [2]:
# Load the calculation being demoed
calculation = iprPy.load_calculation('energy_check')

### 1.2. Display calculation description and theory

In [3]:
# Display main docs and theory
display(Markdown(calculation.maindoc))
display(Markdown(calculation.theorydoc))

# energy_check calculation style

**Lucas M. Hale**, [lucas.hale@nist.gov](mailto:lucas.hale@nist.gov?Subject=ipr-demo), *Materials Science and Engineering Division, NIST*.

Idea suggested by Udo v. Toussaint (Max-Planck-Institute f. Plasmaphysics)

## Introduction

The energy_check calculation style provides a quick check if the energy of an atomic configuration matches with an expected one.

### Version notes

- 6/24/2026 : Method updated to support the LAMMPS library interface.  Shear pressures now also measured.
- 2/25/2025 : Calculation updated to include more outputs and option to create a dumpfile with atomic forces.

### Additional dependencies

### Disclaimers

- [NIST disclaimers](http://www.nist.gov/public_affairs/disclaimer.cfm)

- Small variations in the energy are to be expected due to numerical precisions. 


## Method and Theory

The calculation performs a quick run 0 (no relaxation) energy calculation on a given atomic configuration using a given potential and compares the computed potential energy versus an expected energy value. 

## 2. Display the underlying code

This section displays the underlying code used when calculation.calc() is called in Section 4. It is provided here allowing any users to see and understand how the calculation works.

Feel free to modify and test the functions for yourself!  You can do this by
1. Copy the Python code displayed here into a "Code" cell and run it.
2. In Section 2.2, set "savefiles = True" and run the cell to save any supporting non-python files to the working directory.
3. In Section 4, call the primary calculation function directly rather than using calculation.calc().

### 2.1. Show code and supporting file names and content

In [4]:
# Display calculation code and supporting files
for filename, contents in calculation.files.items():
    display(Markdown(f'## Contents of file "{filename}"'))
    if filename[-3:] == '.py':
        display(Code(contents, language='python'))
    else:
        display(Pretty(contents))

## Contents of file "energy_check.py"

# Python script created by Lucas Hale
# Suggested by Udo v. Toussaint

# Standard library imports
from typing import Optional, Union

# https://github.com/usnistgov/atomman
import atomman as am
import atomman.unitconvert as uc
from atomman.typing import lammpspotential
from atomman.lammps import LAMMPS, LAMMPSobj


def energy_check(lammps_command: Union[str, LAMMPSobj],
                 system: am.System,
                 potential: lammpspotential,
                 mpi_command: Optional[str] = None,
                 dumpforces: bool = False,
                 usefiles: bool = False) -> dict:
    """
    Performs a quick run 0 calculation to evaluate the potential energy of a
    configuration.
    
    Parameters
    ----------
    lammps_command : str, LAMMPSEXE or LAMMPSLIB
        LAMMPS executable command, LAMMPS library name, or an atomman LAMMPS
        interface object.
    system : atomman.System
        The atomic configuration to evaluate.
    potential : atomman.lammps.Potential
        The LAMMPS implemented potential to use.
    mpi_command : str, optional
        The MPI command for running LAMMPS in parallel.  Can only be given if
        lammps_command is a LAMMPS executable command.
    forces : bool, optional
        If True, the atomic forces will also be calculated and returned.
    usefiles : bool, optional
        If set to True, then all input/output files for LAMMPS will be generated.
        Default value of False will minimize the files created.
    
    Returns
    -------
    dict
        Dictionary of results consisting of keys:
        - **'PotEng'** (*float*) - The total potential energy of the system.
        - **'PotEngAtom'** (*float*) - The per-atom potential energy of the system.
        - **'Pxx'** (*float*) - The measured xx component of the pressure on the system.
        - **'Pyy'** (*float*) - The measured yy component of the pressure on the system.
        - **'Pzz'** (*float*) - The measured zz component of the pressure on the system.
        - **'Pxy'** (*float*) - The measured xy component of the pressure on the system.
        - **'Pxz'** (*float*) - The measured xz component of the pressure on the system.
        - **'Pyz'** (*float*) - The measured yz component of the pressure on the system.
        - **'F'** (*numpy.ndarray*) - The atomic forces, returned if forces is True.
    """
    # Create a LAMMPS object if needed
    lmp = LAMMPS(lammps_command, mpi_command=mpi_command, potential=potential)

    # Handle file generation settings
    if usefiles:
        logfile = 'log.lammps'
        script = 'run0.in'
    else:
        logfile = 'none'
        script = None

    # Pass system and potential info into LAMMPS
    lmp.new_system_from_data_file(system, filename='init.dat', tilt_large=True,
                                  usefiles=usefiles, logfile=logfile)

    # Set up thermo style 
    lmp.cmd.thermo_style('custom', 'step', 'pe', 'pxx', 'pyy', 'pzz', 'pxy', 'pxz', 'pyz')
    lmp.cmd.thermo_modify('format', 'float', '%.17e')
    
    # Optionally dump forces to a file
    if usefiles or (not lmp.islib and dumpforces):
        lmp.cmd.dump('dumpy', 'all', 'custom', '1', 'forces.dump', 'id', 'type', ' fx', 'fy', 'fz')
        lmp.cmd.dump_modify('dumpy', 'format', 'float', '%.17e')
    
    # Perform a run 0 to evaluate the system
    lmp.cmd.fix('nve', 'all', 'nve')
    lmp.cmd.run(0)

    # Run EXE versions, get log output
    log = lmp.end_and_get_log(script)

    if log is None:
        # Get thermo directly from lammps object if no log file
        thermo: dict = lmp.last_thermo()
    else:
        # Extract thermo terms from log output
        thermo = log.simulations[0].thermo.iloc[0].to_dict()

    # Convert units on standard thermo terms
    lmp.set_thermo_units(thermo)

    # Add per-atom potential energy
    thermo['PotEngAtom'] = thermo['PotEng'] / system.natoms

    # Get forces directly or from the dump file
    if dumpforces:
        if lmp.islib:
            

### 2.2. Optional: Save supporting files

Set "savefiles = True" to save files locally.  

Note that the code above should be using atomman.tools.read_calc_file() to read the files, which will read any local files with matching names if they exist or read the packaged version if the local files do not exist. This means that if you save the files locally, you can modify them and see how it affects the calculation!

In [5]:
savefiles = False

if savefiles:
    for filename, contents in calculation.files.items():
        if filename[-3:] != '.py':
            with open(filename, 'w') as f:
                f.write(contents)

## 3. Specify input parameters

### 3.1. System-specific paths

- __lammps_command__ is the LAMMPS command to use (required).
- __mpi_command__ MPI command for running LAMMPS in parallel. A value of None will run simulations serially.

In [6]:
lammps_command = 'lmp_serial'
mpi_command = None
#mpi_command = 'mpiexec -localonly 4'

# Optional: check that LAMMPS works and show its version 
print(f'LAMMPS version = {am.lammps.checkversion(lammps_command)["version"]}')

LAMMPS version = 3 Mar 2020


### 3.2. Interatomic potential

- __potential_name__ gives the name of a potential_LAMMPS record to find and download from the iprPy library.  
- __potential__ is a potential_LAMMPS or potential_LAMMPS_KIM record object (required).

See documentation for the [potentials package](https://github.com/usnistgov/potentials/tree/master/doc) for more options on finding, loading and building potential objects (doc Notebook #s 0, 5.3, 5.4 and 7).

In [7]:
potential_name = '1999--Mishin-Y--Ni--LAMMPS--ipr1'

# Retrieve potential and parameter file(s) using atomman
potential = am.load_lammps_potential(id=potential_name, getfiles=True)

### 3.3. System

- __system__ is an atomman.System representing a fundamental unit cell of the system (required).  Here, this is loaded as the ucell from a relaxed_crystal record.
- __expected_potential_energy__ is the expected per-atom potential energy for the system.  Not needed for the calculation itself, but used here to compare with the computed value.  This is taken from the relaxed_crystal record.

See documentation for the [atomman package](https://github.com/lmhale99/atomman/tree/master/doc/tutorial) for more options on building and loading atomic configurations (doc Notebook #s 1.1, 1.2, 1.3, 1.4 and 1.4.*)

In [8]:
# Fetch a relaxed crystal record from the database
potdb = am.library.Database()
crystal = potdb.get_relaxed_crystal(potential_LAMMPS_id=potential.id, family='A1--Cu--fcc', standing='good')

# Set ucell from the crystal record
system = crystal.ucell

# Set the expected potential energy from the crystal record
expected_potential_energy = crystal.potential_energy

Multiple matching record retrieved from remote
#  family               symbols  alat    Ecoh    method  standing
 1 A1--Cu--fcc          Ni        3.5200 -4.4500 dynamic good
 2 A1--Cu--fcc          Ni        7.3760  0.0119 dynamic good


Please select one: 1


## 4. Run calculation and view results

### 4.1. Run calculation

All primary calculation method functions take a series of inputs and return a dictionary of outputs.

In [9]:
# What is calculation.calc an alias of?
calculation.calc.__module__

'iprPy.calculation.energy_check.energy_check'

In [10]:
# Run and display results
results_dict = calculation.calc(lammps_command, system, potential, mpi_command=mpi_command, dumpforces=True)
print(results_dict.keys())

dict_keys(['Step', 'PotEng', 'Pxx', 'Pyy', 'Pzz', 'Pxy', 'Pxz', 'Pyz', 'PotEngAtom', 'F'])


### 4.2. Report results

Values returned in the results_dict:

- 'Step' is the final relaxation step, should be 0.
- 'PotEng' is the computed total potential energy across all atoms.
- 'PotEngAtom' is the computed average potential energy across all atoms.
- 'Pxx' is the computed xx component of pressure on the system.
- 'Pyy' is the computed yy component of pressure on the system.
- 'Pzz' is the computed zz component of pressure on the system.
- 'Pxy' is the computed xy component of pressure on the system.
- 'Pxz' is the computed xz component of pressure on the system.
- 'Pyz' is the computed yz component of pressure on the system.
- 'F' are the per-atom forces, included only if dumpforces=True.

In [11]:
energy_unit = 'eV'
print('Measured potential energy:', uc.get_in_units(results_dict['PotEngAtom'], energy_unit), energy_unit)
if expected_potential_energy is not None:
    print('Expected potential energy:', uc.get_in_units(expected_potential_energy, energy_unit), energy_unit)

Measured potential energy: -4.449999998348881 eV
Expected potential energy: -4.44999999835575 eV


In [12]:
pressure_unit = 'GPa'
print('Measured Pxx:', uc.get_in_units(results_dict['Pxx'], pressure_unit), pressure_unit)
print('Measured Pyy:', uc.get_in_units(results_dict['Pyy'], pressure_unit), pressure_unit)
print('Measured Pzz:', uc.get_in_units(results_dict['Pzz'], pressure_unit), pressure_unit)
print('Measured Pxy:', uc.get_in_units(results_dict['Pxy'], pressure_unit), pressure_unit)
print('Measured Pxz:', uc.get_in_units(results_dict['Pxz'], pressure_unit), pressure_unit)
print('Measured Pyz:', uc.get_in_units(results_dict['Pyz'], pressure_unit), pressure_unit)

Measured Pxx: -1.4933463206139311e-09 GPa
Measured Pyy: -1.4933444853213094e-09 GPa
Measured Pzz: -1.493344571350651e-09 GPa
Measured Pxy: -9.240188548860662e-17 GPa
Measured Pxz: 3.441173666472246e-16 GPa
Measured Pyz: 1.9117631480401373e-16 GPa


In [13]:
force_unit = 'eV/angstrom'
if 'F' in results_dict:
    print(f'Measured forces (in {force_unit}):')
    print(uc.get_in_units(results_dict['F'], force_unit))

Measured forces (in eV/angstrom):
[[-2.20413965e-14 -2.20379270e-14 -2.20145083e-14]
 [-2.20344576e-14  2.20240493e-14  2.20123399e-14]
 [ 2.18852714e-14 -2.20309881e-14  2.20361923e-14]
 [ 2.19962937e-14  2.20223145e-14 -2.19702728e-14]]


### 4.3. Optional: Clean calculation files

The calculation may generate output files when it runs.  Calling calculation.clean_files() will delete any generated files to keep the workspace clean.

In [14]:
calculation.clean_files()